# Duplicates: exact, fuzzy, and the leak across the split

_Data Wrangling_

**Find rows that say the same thing twice — both perfect copies and the same entity spelled a little differently — and collapse them before you split.**

A duplicate is a row that repeats information already present in another row. The danger
       is that duplicates are invisible to most code &mdash; the table still has clean rows and columns,
       it just secretly over-counts some records. That quietly corrupts everything downstream:
       counts, averages, class balance, and &mdash; worst &mdash; the honesty of your test set.

       There are two regimes, and they need different tools:

---

This notebook builds the de-duplication workflow one stage at a time — exact, then fuzzy, then the split. Run each cell, read the note above it, then tackle the practice problems at the bottom. _Save a copy to your Drive (File → Save a copy in Drive) to edit and keep your work._

In [ ]:
# Setup — numpy / pandas / scikit-learn / matplotlib ship with Colab.
!pip install -q rapidfuzz
import numpy as np
import matplotlib.pyplot as plt

## Reference implementation — pandas + rapidfuzz

### Step 1 — Build a customer table with planted duplicates

We start with a customer table merged from two systems. Some rows are **exact** copies, some are **fuzzy** near-duplicates (a misspelled name, extra spaces, different casing), and a couple — like "Robert Lee" vs "Bob Lee" — are genuinely distinct people we must keep. The `id` column is bookkeeping only; the real identity lives in `name`, `city`, and `signup`.

In [ ]:
import pandas as pd
from rapidfuzz import fuzz

# --- A customer table merged from two systems, with planted duplicates ---
df = pd.DataFrame([
    {"id": 1,  "name": "John Smith",    "city": "Boston",   "signup": "2024-01-05"},
    {"id": 2,  "name": "John Smith",    "city": "Boston",   "signup": "2024-01-05"},  # exact dup
    {"id": 3,  "name": "Jon Smith",     "city": "Boston",   "signup": "2024-01-05"},  # fuzzy
    {"id": 4,  "name": "John  Smith ",  "city": "boston",   "signup": "2024-01-05"},  # fuzzy
    {"id": 5,  "name": "Mary Johnson",  "city": "New York", "signup": "2024-02-11"},
    {"id": 6,  "name": "Mary Johnson",  "city": "New York", "signup": "2024-02-11"},  # exact dup
    {"id": 7,  "name": "Mary Jonson",   "city": "New York", "signup": "2024-02-11"},  # fuzzy
    {"id": 8,  "name": "Alice Wong",    "city": "Seattle",  "signup": "2024-03-02"},
    {"id": 9,  "name": "alice wong",    "city": "Seattle",  "signup": "2024-03-02"},  # fuzzy (case)
    {"id": 10, "name": "Bob Lee",       "city": "Chicago",  "signup": "2024-04-19"},
    {"id": 11, "name": "Bob Lee",       "city": "Chicago",  "signup": "2024-04-19"},  # exact dup
    {"id": 12, "name": "Robert Lee",    "city": "Chicago",  "signup": "2024-04-19"},  # distinct -> keep
    {"id": 13, "name": "Carla Diaz",    "city": "Miami",    "signup": "2024-05-08"},
    {"id": 14, "name": "Carla Diaz",    "city": "Miami",    "signup": "2024-05-08"},  # exact dup
])
print("raw rows:", len(df))                          # 14

### Step 2 — Drop exact duplicates on the identity columns

Exact duplicates are rows identical across the columns that define identity. We pass `subset=["name", "city", "signup"]` so the bookkeeping `id` is ignored — otherwise every row would look unique. `keep="first"` retains the first copy of each group and drops the rest. This collapses the 14 raw rows down to 10.

In [ ]:
# === 1) EXACT duplicates on the identity columns (ignore the bookkeeping 'id') ===
keys = ["name", "city", "signup"]
print("exact dups:", df.duplicated(subset=keys).sum())   # 4

df = df.drop_duplicates(subset=keys, keep="first")       # keep the first copy
print("after exact drop:", len(df))                      # 10

### Step 3 — Collapse fuzzy near-duplicates: normalize, block, match

Fuzzy duplicates survive the exact pass because their text differs slightly. The recipe is: **normalize** each name (lowercase, strip, collapse internal spaces), **block** so we only compare rows sharing the same city and signup date, then **match** with rapidfuzz's similarity score (0..100) against a threshold `TAU`. Each matched row is pointed at a canonical record, and we keep only the canonical rows — leaving 6 distinct entities. Crucially, this must run **before** any train/test split, or duplicates leak across it.

In [ ]:
# === 2) FUZZY / near-duplicates: normalize -> block -> match -> collapse ===
def norm(s):                                # lowercase, strip, collapse internal spaces
    return " ".join(str(s).lower().split())
df["nkey"] = df["name"].map(norm)

TAU = 85                                     # rapidfuzz score is 0..100; merge if >= TAU
canon = {}                                    # row index -> canonical row index
rows = df.reset_index(drop=True)
for i in range(len(rows)):
    if i in canon:
        continue
    canon[i] = i                              # i is its own canonical record
    for j in range(i + 1, len(rows)):
        if j in canon:
            continue
        # BLOCK: only compare rows in the same city + signup date
        same_block = (rows.at[i, "city"].lower() == rows.at[j, "city"].lower()
                      and rows.at[i, "signup"] == rows.at[j, "signup"])
        if same_block and fuzz.ratio(rows.at[i, "nkey"], rows.at[j, "nkey"]) >= TAU:
            canon[j] = i                      # j is a near-dup of i

clean = rows.loc[[i for i in range(len(rows)) if canon[i] == i]].drop(columns="nkey")
print("after fuzzy collapse:", len(clean))   # 6 distinct entities

# NOTE: do all of the above BEFORE train_test_split, or duplicates leak across the split.

## Visualize the data & results

_Starting from 14 customer rows merged from two systems (with planted dupes), how many rows survive after dropping EXACT duplicates, and after also collapsing FUZZY near-duplicates?_

### Step 4 — Rebuild the table and run the exact pass

To see how the row count shrinks at each stage, we rebuild the same 14-row table (this time without the `id` column) and re-run the exact de-dup. As before, dropping exact duplicates on `name`, `city`, and `signup` takes us from 14 rows down to 10.

In [ ]:
import pandas as pd
from rapidfuzz import fuzz
from itertools import combinations

df = pd.DataFrame([
    {"name":"John Smith",   "city":"Boston",  "signup":"2024-01-05"},
    {"name":"John Smith",   "city":"Boston",  "signup":"2024-01-05"},
    {"name":"Jon Smith",    "city":"Boston",  "signup":"2024-01-05"},
    {"name":"John  Smith ", "city":"boston",  "signup":"2024-01-05"},
    {"name":"Mary Johnson", "city":"New York","signup":"2024-02-11"},
    {"name":"Mary Johnson", "city":"New York","signup":"2024-02-11"},
    {"name":"Mary Jonson",  "city":"New York","signup":"2024-02-11"},
    {"name":"Alice Wong",   "city":"Seattle", "signup":"2024-03-02"},
    {"name":"alice wong",   "city":"Seattle", "signup":"2024-03-02"},
    {"name":"Bob Lee",      "city":"Chicago", "signup":"2024-04-19"},
    {"name":"Bob Lee",      "city":"Chicago", "signup":"2024-04-19"},
    {"name":"Robert Lee",   "city":"Chicago", "signup":"2024-04-19"},
    {"name":"Carla Diaz",   "city":"Miami",   "signup":"2024-05-08"},
    {"name":"Carla Diaz",   "city":"Miami",   "signup":"2024-05-08"},
])
n_raw = len(df)                                              # 14

# Exact pass
keys = ["name","city","signup"]
df_exact = df.drop_duplicates(subset=keys, keep="first")    # 14 -> 10
n_exact = len(df_exact)

### Step 5 — Sweep the fuzzy threshold to see its effect

Here we wrap the fuzzy collapse in a function and run it at several thresholds. It uses union-find (`parent`/`find`) to merge matched rows into entities, with the same normalize-and-block logic as the reference (note the score is compared against `tau*100` since `tau` here is on a 0..1 scale). Sweeping `tau` shows the tradeoff: a loose threshold (0.70) over-merges into 5 entities, while a strict one (0.95) under-merges into 7.

In [ ]:
# Fuzzy pass: normalize, block on (city,date), union-find merge above threshold
norm = lambda s: " ".join(str(s).lower().split())
recs = df_exact.assign(nkey=df_exact["name"].map(norm)).reset_index(drop=True)

def n_entities(tau):
    parent = list(range(len(recs)))
    def find(x):
        while parent[x]!=x: parent[x]=parent[parent[x]]; x=parent[x]
        return x
    for i,j in combinations(range(len(recs)),2):
        block = (recs.at[i,"city"].lower()==recs.at[j,"city"].lower()
                 and recs.at[i,"signup"]==recs.at[j,"signup"])
        if block and fuzz.ratio(recs.at[i,"nkey"], recs.at[j,"nkey"]) >= tau*100:
            parent[find(i)] = find(j)
    return len({find(i) for i in range(len(recs))})

print(n_raw, n_exact, n_entities(0.85))                     # 14 10 6
print([n_entities(t) for t in (0.70,0.80,0.85,0.90,0.95)])  # [5, 6, 6, 6, 7]

## Practice

Try each one in the empty cell below it, then reveal the worked solution.

**Problem 1.** You merged two customer exports and want to remove duplicate accounts. df.drop_duplicates() (no arguments) removes nothing, yet you can see the same customer twice. What is happening and how do you fix it?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Check whether the rows are truly identical across all columns. — _drop_duplicates() with no subset only drops rows identical in every column; one differing column (a row id, an export timestamp, a source tag) makes them "distinct"._
- Pick the columns that actually define a customer &mdash; e.g. ['name','email']. — _That is the identity key; differing bookkeeping columns should not count._
- Run df.drop_duplicates(subset=['name','email'], keep='first') and inspect what was dropped. — _Restricting to the identity subset collapses the real dupes; inspecting guards against deleting genuinely distinct rows._

**Answer:** A bookkeeping column (id, timestamp, source) differs, so the rows aren't byte-identical and the no-argument call keeps them. Pass an explicit subset of the columns that define identity &mdash; df.drop_duplicates(subset=['name','email'], keep='first') &mdash; and check the dropped rows to be sure you didn't lose real records.

</details>

**Problem 2.** Your fuzzy de-dup with rapidfuzz at threshold $\tau=0.70$ merged "Bob Lee" with "Robert Lee" and two genuinely different "Lin Chen"s in the same city. Is the threshold too loose or too tight, and what two changes help?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Recognize that distinct entities are being fused. — _Fusing distinct entities is the symptom of a threshold that is too loose (too low)._
- Raise $\tau$ (e.g. to 0.85&ndash;0.90). — _A stricter cutoff stops merging pairs that are only moderately similar, like "Bob"/"Robert"._
- Add or tighten a blocking key and add a second compared field. — _Blocking on city + signup date and also comparing email/phone means two different "Lin Chen"s won't match unless several fields agree._

**Answer:** Too loose &mdash; a low $\tau$ glues distinct people together. Raise the threshold (toward 0.85&ndash;0.90) and strengthen blocking (compare only rows sharing city/date) while matching on more than one field (name and email/phone). That stops false merges without losing the real "Jon"/"John" dupes.

</details>

**Problem 3.** A teammate splits into train/test first, then de-duplicates each split separately. Why is this the wrong order, and what is the correct order?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Note that an exact or near-duplicate pair can be split with one copy in train and one in test. — _Splitting before de-duping lets the same record land on both sides._
- See that the model then trains on (essentially) a test row. — _That is leakage: the test score reflects memorization, not generalization, so it is inflated._
- De-duplicate the full table first (exact, then fuzzy), then split. — _Each distinct entity exists once and lands entirely on one side, keeping the test set honest._

**Answer:** De-duping after the split can't catch a pair whose two copies already sit on opposite sides &mdash; the duplicate has already leaked across, inflating the test score. Always de-duplicate the whole dataset before splitting, so every real entity appears once and on a single side.

</details>